# Fixations to Bottom-Strip
Determine the value of hyperparameter `FIXATIONS_TO_STRIP_THRESHOLD`, which is used to decide if a fixation / visit is a _LWS_ instance or not by excluding _LWS_ candidates that occur too close to a fixation in the bottom-strip (where exemplars are presented).

##### Update:
There's not much difference if this value is set to 2/3/4 fixations, so we can choose `FIXATIONS_TO_STRIP_THRESHOLD = 3` fixations as a reasonable compromise.

In [1]:
import numpy as np
import pandas as pd
import plotly.io as pio

import config as cnfg
from data_models.SearchArray import SearchArray
from plgrnd2 import fixations, actions

pio.renderers.default = "notebook"  # or "browser"

### Read data

In [2]:
from analysis.helpers.read_data import read_data

loaded_data = read_data(cnfg.OUTPUT_PATH)
actions = loaded_data.actions
idents = loaded_data.identifications
fixations = loaded_data.fixations
del loaded_data

In [3]:
fixs_with_ident_time = fixations.copy()
dva_cols = [col for col in fixations.columns if col.endswith("distance_dva")]
fixs_with_ident_time["target"] = fixs_with_ident_time[dva_cols].idxmin(axis=1).str.replace("_distance_dva", "")
fixs_with_ident_time = (
    fixs_with_ident_time
    .drop(columns=[col for col in fixs_with_ident_time.columns if "_distance_" in col])
    .merge(
        idents.loc[
            idents["identification_category"] == "hit", ["subject", "trial", "target", "time"]
        ], on=["subject", "trial", "target"], how="left"
    )
)

fixs_with_ident_time.loc[:, "is_during"] = (fixs_with_ident_time["start_time"] <= fixs_with_ident_time["time"]) & (fixs_with_ident_time["time"] <= fixs_with_ident_time["end_time"])

fixs_with_ident_time.loc[:, "is_in_strip"] = fixs_with_ident_time.apply(
    lambda row: SearchArray.is_in_bottom_strip(tuple([row["x"], row["y"]])), axis=1
)

##### Count how many fixations are between a bottom-strip fixation and a target-identification fixation:

In [4]:
def count_fixations_from_strip_to_identification(is_in_strip: pd.Series, is_during: pd.Series) -> pd.Series:
    """
    For a single trial's fixations, count how many fixations are between a bottom-strip fixation and a target-identification fixation.
    For non-bottom-strip fixations, the count is NaN.
    """
    assert len(is_in_strip) == len(is_during), "is_in_strip and is_during must have the same length"
    assert is_in_strip.index.equals(is_during.index), "is_in_strip and is_during must have the same index"
    num_fixs_between = pd.Series(np.nan, index=is_during.index, dtype=float)
    during_indices = np.flatnonzero(is_during.values)
    if len(during_indices) == 0:
        return num_fixs_between     # no identification fixations - return NaN for all fixations
    row_indices = np.arange(len(is_during))
    next_during_indices = np.searchsorted(during_indices, row_indices, side="right")
    has_subsequent_ident = next_during_indices < len(during_indices)
    ident_distances = np.full_like(is_during, np.nan, dtype=float)
    ident_distances[has_subsequent_ident] = during_indices[next_during_indices[has_subsequent_ident]] - row_indices[has_subsequent_ident]    # distance between each fixation and the next identification fixation
    num_fixs_between[is_in_strip] = ident_distances[is_in_strip]    # apply only to bottom-strip fixations
    return num_fixs_between


# populate the dataframe
fixs_with_ident_time["num_fixs_strip_to_ident"] = np.nan
for (subj_id, trial_num, eye), data in fixs_with_ident_time.groupby(["subject", "trial", "eye"]):
    data = data.sort_values("start_time")
    num_fixs_to_strip = count_fixations_from_strip_to_identification(
        data["is_in_strip"], data["is_during"]
    )
    fixs_with_ident_time.loc[data.index, "num_fixs_strip_to_ident"] = num_fixs_to_strip

### Number of Fixations until Bottom-Strip Visit
#### (1) All Fixations

In [5]:
valid_fixs_with_ident_time = fixs_with_ident_time.loc[np.isfinite(fixs_with_ident_time["num_fixs_to_strip"])]

fixs_summary = (
    pd.concat([
        valid_fixs_with_ident_time["num_fixs_to_strip"].describe().rename("all"),
        valid_fixs_with_ident_time.groupby("subject")["num_fixs_to_strip"].describe().T,
    ], axis=1)
).T

print("All Fixations:")
fixs_summary

All Fixations:


,count,mean,std,min,25%,50%,75%,max
all,37655.0,21.220794,20.315359,0.0,6.0,15.0,30.0,126.0
2,3003.0,24.374958,21.758178,0.0,8.0,19.0,35.0,126.0
12,3497.0,18.310838,17.184864,0.0,5.0,14.0,26.0,104.0
13,2533.0,12.506514,11.414133,0.0,4.0,9.0,18.0,68.0
14,2580.0,15.399225,15.251654,0.0,4.0,11.0,22.0,84.0
15,3699.0,20.854015,19.501683,0.0,6.0,15.0,30.0,98.0
16,2959.0,15.444407,13.535302,0.0,5.0,12.0,22.5,76.0
17,1436.0,13.487465,11.321614,0.0,4.0,11.0,20.0,55.0
18,4379.0,32.291847,26.063470,0.0,10.0,26.0,50.0,100.0
19,2564.0,11.215289,11.205336,0.0,3.0,8.0,16.0,65.0


#### (2) Ident Fixations

In [6]:
ident_fixs = fixs_with_ident_time.loc[fixs_with_ident_time["is_during"]]
valid_ident_fixs_with_ident_time = ident_fixs.loc[np.isfinite(fixs_with_ident_time["num_fixs_to_strip"])]

fixs_summary = (
    pd.concat([
        valid_ident_fixs_with_ident_time["num_fixs_to_strip"].describe().rename("all"),
        valid_ident_fixs_with_ident_time.groupby("subject")["num_fixs_to_strip"].describe().T,
    ], axis=1)
).T

print("Identification Fixations:")
fixs_summary

Identification Fixations:


,count,mean,std,min,25%,50%,75%,max
all,798.0,18.710526,20.154053,1.0,4.00,12.0,25.0,104.0
2,58.0,25.603448,23.055950,1.0,8.25,20.0,38.0,91.0
12,55.0,16.836364,15.284661,1.0,4.50,13.0,26.5,75.0
13,63.0,12.253968,11.579972,1.0,4.00,9.0,17.5,60.0
14,64.0,10.812500,10.096165,1.0,2.00,8.0,16.0,46.0
15,91.0,12.285714,15.605046,1.0,2.50,6.0,17.0,74.0
16,57.0,17.754386,13.732496,1.0,10.00,14.0,21.0,74.0
17,30.0,14.400000,12.090464,1.0,5.25,11.0,18.0,52.0
18,89.0,29.056180,25.748945,1.0,8.00,20.0,48.0,97.0
19,81.0,7.320988,8.391703,1.0,2.00,4.0,10.0,46.0


### Number of Fixations from Bottom-Strip to Identification

In [7]:
finite_num_from_strip_to_iden = fixs_with_ident_time.loc[np.isfinite(fixs_with_ident_time["num_fixs_strip_to_ident"])]

percentiles = [0.01, 0.05, 0.1, 0.25, 0.5]
fixs_summary = (
    pd.concat([
        finite_num_from_strip_to_iden["num_fixs_strip_to_ident"].describe(percentiles).rename("all"),
        finite_num_from_strip_to_iden.groupby("subject")["num_fixs_strip_to_ident"].describe(percentiles).T,
    ], axis=1)
).T

print("From Strip to Identification:")
fixs_summary

From Strip to Identification:


,count,mean,std,min,1%,5%,10%,25%,50%,max
all,1346.0,19.198366,16.466165,1.0,1.00,2.00,3.0,6.00,14.0,111.0
2,67.0,16.432836,13.963398,1.0,1.00,2.00,3.0,5.00,12.0,57.0
12,131.0,23.580153,20.442021,1.0,1.00,2.50,4.0,7.00,18.0,111.0
13,125.0,14.592000,12.950960,2.0,2.00,2.00,2.0,4.00,10.0,65.0
14,96.0,21.125000,14.514059,1.0,1.95,2.75,4.0,10.75,17.5,63.0
15,159.0,19.000000,16.677507,1.0,1.00,2.00,3.0,6.00,13.0,70.0
16,116.0,21.120690,18.069987,1.0,1.15,2.00,3.0,6.00,16.5,70.0
17,69.0,14.855072,13.229063,1.0,1.00,2.00,2.0,3.00,12.0,46.0
18,109.0,25.256881,17.756274,1.0,2.00,2.00,3.0,10.00,24.0,71.0
19,136.0,17.602941,15.502105,1.0,1.35,2.00,3.0,6.00,13.0,68.0
